# Quantcore Python API demo

This notebook exercises the API for the four researcher roles and the Quant Developer. It uses the fuller NOX catalog explicitly. Rust remains the computation backend; Nautilus Trader remains responsible for orders, positions, and fills.

In [ ]:
from pathlib import Path
import math
import sys

ROOT = Path("/Users/ducle/repos/quant_core")
NOX_CATALOG = Path("/Users/ducle/repos/nox_system/data/catalog")
assert ROOT.exists(), ROOT
assert NOX_CATALOG.exists(), NOX_CATALOG
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quantcore.core import DataConfig, load_bars

data = DataConfig(
    catalog_path=str(NOX_CATALOG),
    start="2026-07-15",
    end="2026-08-29",
)
bars = load_bars(data)
print(f"{len(bars):,} bars: {bars.ts.min()} -> {bars.ts.max()}")
bars.head(3)

## 1. Quantitative Researcher

`evaluate_seed` is the Rust DSL evaluation workflow. Custom score models use the `QuantitativeModel` protocol and `score_model`; they are not assigned a fake DSL identity.

In [ ]:
from quantcore.alpha import (
    AlphaConfig,
    quantitative_models,
    evaluate_seed,
    score_model,
)

tear_sheet = evaluate_seed(
    "close - ewma(close, 8)",
    config=AlphaConfig(data=data),
    record_trial=False,
)
print({
    "alpha_id": tear_sheet.alpha_id,
    "verdict": tear_sheet.verdict,
    "net_sharpe": tear_sheet.metrics["net_sharpe"],
    "reasons": tear_sheet.reasons,
})

class VolumeAdjustedMomentum:
    def score(self, close, volume):
        result = [0.0]
        for index in range(1, len(close)):
            ret = close[index] / close[index - 1] - 1.0
            result.append(ret * math.log1p(max(volume[index], 0.0)))
        return result

quantitative_models.register(
    "notebook-volume-momentum",
    VolumeAdjustedMomentum(),
    replace=True,  # makes Run All repeatable
)
close = bars["close"].astype(float).tolist()
volume = bars["volume"].astype(float).tolist()
custom_scores = score_model("notebook-volume-momentum", close, volume)
print(f"custom model returned {len(custom_scores):,} aligned scores")

## 2. Portfolio Researcher

`combine` accepts aligned score series. A custom `PortfolioOptimizer` implements `optimize(scores)`.

In [ ]:
from quantcore.portfolio import combine, combine_methods

score_set = {
    "engine-close": score_model("engine_close", close, volume),
    "volume-momentum": custom_scores,
}
built_in_portfolio = combine(score_set, method="inverse_vol")
print("inverse-volatility weights:", built_in_portfolio["weights"])

class EqualBlend:
    def optimize(self, score_rows):
        count = len(score_rows)
        return [sum(values) / count for values in zip(*score_rows)]

combine_methods.register("notebook-equal-blend", EqualBlend(), replace=True)
custom_portfolio = combine(score_set, method="notebook-equal-blend")
composite = custom_portfolio["composite"]
print(custom_portfolio["provenance"], len(composite))

## 3. Risk Researcher

A custom `RiskMeasure` runs through the same `backtest_portfolio` workflow as the Rust-backed sizing methods.

In [ ]:
from quantcore.risk import (
    RiskBacktestConfig,
    backtest_portfolio,
    sizing_methods,
)

class HalfExposure:
    def adjust(self, z_scores, vol_est, target_vol, drawdowns=None, floor=None):
        return [0.5 * value for value in z_scores]

sizing_methods.register("notebook-half-exposure", HalfExposure(), replace=True)
risk_result = backtest_portfolio(
    composite,
    data=data,
    sizing="notebook-half-exposure",
    policy="trigger_matrix",
    config=RiskBacktestConfig(data=data),
)
print("before:", risk_result["before"]["performance"])
print("after risk:", risk_result["after"]["risk_process"])

## 4. Execution Researcher

`plan_orders` validates every custom plan before the Nautilus-backed execution workflow consumes it.

In [ ]:
import pandas as pd
from quantcore.execution import (
    ExecutionConfig,
    backtest_execution,
    execution_algorithms,
    plan_orders,
)

class TwoSlice:
    def plan(self, gap_contracts, config):
        first = gap_contracts // 2
        second = gap_contracts - first
        return [quantity for quantity in (first, second) if quantity]

execution_algorithms.register("notebook-two-slice", TwoSlice(), replace=True)
print("custom plan:", plan_orders(5, "notebook-two-slice", ExecutionConfig(slice_bars=2)))

targets = pd.DataFrame({
    "ts": bars.iloc[[50, 100, 150]]["ts"].tolist(),
    "target_contracts": [2, -1, 0],
})
execution_result = backtest_execution(
    targets,
    config=ExecutionConfig(cooldown_secs=0.0, min_gap_contracts=1),
    data=data,
    algo="notebook-two-slice",
)
print({key: execution_result[key] for key in ("n_orders", "n_fills", "n_rejected", "algo")})

## Shared handoff: Portfolio → Risk → Execution

`TargetPosition` is desired exposure. `RiskDecision.approved_target_contracts` is the only target execution may pursue. Nautilus owns the actual position.

In [ ]:
from datetime import datetime, timezone
from trading.contracts import RiskDecision, TargetPosition

desired = TargetPosition(
    ts=datetime.now(timezone.utc),
    target_contracts=5,
    z_target=1.1,
    reason="signal",
)
risk_decision = RiskDecision(
    desired=desired,
    approved_target_contracts=2,
    action="cap",
    reason="exposure-limit",
    current_contracts=1,
)
risk_decision

## 5. Quant Developer

Paper and live both use the Nautilus live runtime. Paper selects the Entrade demo account; live selects the Entrade live account. The following builds configuration only and does not connect.

In [ ]:
from nautilus_trader.common import Environment
from trading.instruments import load_futures_instrument_spec
from trading.node import (
    ExecutionBroker,
    TradingEnvironment,
    TradingRuntimeConfig,
    build_trading_node_config,
)

instrument = load_futures_instrument_spec(
    ROOT / "src/market_data/instrument_definitions/vn30f1m.hnx.json"
)
paper_runtime = TradingRuntimeConfig(
    api_key="<API_KEY>",
    api_secret="<API_SECRET>",
    instrument_spec=instrument,
    alpha_path="data/pool/weights.json",
    execution_broker=ExecutionBroker.ENTRADE,
    execution_environment=TradingEnvironment.DEMO,
    max_exposure_contracts=3,
)
node_config = build_trading_node_config(paper_runtime)
entrade_config = next(iter(node_config.exec_clients.values()))
assert node_config.environment == Environment.LIVE
print({
    "nautilus_runtime": node_config.environment.value,
    "entrade_account": entrade_config.environment.value,
    "connected": False,
})